# 00 Estimate Linear Lambda Init

Runs tuned LinearBidder on the train split and estimates the constant DRLB lambda init as `ctr_pred / bid`.

In [ ]:
import sys
import pickle
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.linear_lambda import estimate_linear_lambda_init
from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.infra.split_utils import resolve_normalized_splits


In [ ]:
# Tuned linear params live under evaluate_baselines/best_params/<subfolder>/ (see baselines_finetune.BaseLineTrainer).
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
# Last expression must be at module level - Jupyter does not auto-display values inside with / if / etc.
linear_tuned_params


In [ ]:
config_ref = build_drlb_config(
    run_name='may05_estimate_linear_lambda_init',
    profile='may05_default_best_fixed',
    split_set='full_train_val_holdout',
)
normalized_splits = resolve_normalized_splits(config_ref)
linear_params = {
    'cold_start_coef': linear_tuned_params['coef'],
    'lower_clip': linear_tuned_params['lower_clip'],
    'upper_clip': linear_tuned_params['upper_clip'],
    'factor': linear_tuned_params['factor'],
}
linear_params


In [ ]:
linear_lambda_info = estimate_linear_lambda_init(
    normalized_splits=normalized_splits,
    linear_params=linear_params,
    auction_mode=config_ref.auction_mode,
)
linear_lambda_init = linear_lambda_info['linear_lambda_init']
linear_lambda_median = linear_lambda_info['linear_lambda_median']
linear_lambda_summary = linear_lambda_info['linear_lambda_summary']
linear_lambda_init, linear_lambda_median, linear_lambda_summary


In [ ]:
linear_lambda_info['linear_lambda_examples']
